In [1]:
import os
import csv
import json
import time
from datetime import date

import requests
from dotenv import load_dotenv
from rapidfuzz import fuzz, process

load_dotenv()

API_KEY = os.getenv("OMDB_API_KEY")
if not API_KEY:
    raise SystemExit("Set OMDB_API_KEY in a .env file next to this script before running.")

SOURCE_CSV = "IMDB (1980-2020).csv"
ENRICHED_CSV = "imdb_enriched.csv"
UNMATCHED_CSV = "omdb_unmatched.csv"
PROGRESS_FILE = "omdb_progress.json"

# Leave a small buffer under the real ~1000/day cap so we stop
# gracefully instead of hitting a hard error mid-request.
DAILY_REQUEST_BUDGET = 950
REQUEST_TIMEOUT = 10
SLEEP_BETWEEN_REQUESTS = 0.25  # be a polite API citizen

FIELDNAMES_OUT = [
    "name", "year", "match_method", "omdb_title", "omdb_year",
    "Plot", "Genre_full", "Actors_full", "Writer_full",
    "Director_full", "imdbRating_omdb",
]


def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            state = json.load(f)
        # reset the daily counter if it's a new day
        if state.get("date") != str(date.today()):
            state["requests_today"] = 0
            state["date"] = str(date.today())
        return state
    return {"last_index": -1, "requests_today": 0, "date": str(date.today())}


def save_progress(state):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(state, f)


def omdb_get(params):
    params = {**params, "apikey": API_KEY}
    resp = requests.get("http://www.omdbapi.com/", params=params, timeout=REQUEST_TIMEOUT)
    return resp.json()

def exact_lookup(title, year):
    return omdb_get({"t": title, "y": int(year), "type": "movie", "plot": "full"})


def fuzzy_fallback(title, year):
    search_result = omdb_get({"s": title, "type": "movie"})
    if is_rate_limited(search_result):
        return search_result  # pass the rate-limit signal up, don't swallow it
    if search_result.get("Response") != "True":
        return None

    candidates = search_result.get("Search", [])
    if not candidates:
        return None

    titles = [c["Title"] for c in candidates]
    best = process.extractOne(title, titles, scorer=fuzz.token_sort_ratio)
    if best is None or best[1] < 70:
        return None

    matched_title, score, idx = best
    candidate = candidates[idx]

    same_title_candidates = [c for c in candidates if c["Title"] == matched_title]
    if len(same_title_candidates) > 1:
        year_matches = [c for c in same_title_candidates if c.get("Year", "").startswith(str(year))]
        if year_matches:
            candidate = year_matches[0]

    return omdb_get({"i": candidate["imdbID"], "type": "movie", "plot": "full"})


def is_rate_limited(data):
    if data is None:
        return False
    return data.get("Response") == "False" and "limit" in data.get("Error", "").lower()


def main():
    with open(SOURCE_CSV, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    state = load_progress()
    start_index = state["last_index"] + 1

    if start_index >= len(rows):
        print("All rows already processed. Nothing to do.")
        return

    enriched_file_exists = os.path.exists(ENRICHED_CSV)
    unmatched_file_exists = os.path.exists(UNMATCHED_CSV)

    enriched_f = open(ENRICHED_CSV, "a", newline="", encoding="utf-8")
    unmatched_f = open(UNMATCHED_CSV, "a", newline="", encoding="utf-8")
    enriched_writer = csv.DictWriter(enriched_f, fieldnames=FIELDNAMES_OUT)
    unmatched_writer = csv.DictWriter(unmatched_f, fieldnames=["name", "year", "reason"])
    if not enriched_file_exists:
        enriched_writer.writeheader()
    if not unmatched_file_exists:
        unmatched_writer.writeheader()

    print(f"Resuming from row {start_index}/{len(rows)}. "
          f"{state['requests_today']}/{DAILY_REQUEST_BUDGET} requests used today.")

    processed_this_run = 0
    for i in range(start_index, len(rows)):
        if state["requests_today"] >= DAILY_REQUEST_BUDGET:
            print(f"Hit today's request budget ({DAILY_REQUEST_BUDGET}). "
                  f"Stopping cleanly at row {i}. Run the script again "
                  f"tomorrow to continue.")
            break

        row = rows[i]
        title, year = row["name"], row["year"]

        data = exact_lookup(title, year)
        state["requests_today"] += 1
        method = "exact"

        if is_rate_limited(data):
            print("OMDb reports daily limit reached. Stopping cleanly.")
            state["last_index"] = i - 1
            save_progress(state)
            break

        if data.get("Response") != "True":
            if state["requests_today"] >= DAILY_REQUEST_BUDGET:
                break
            data = fuzzy_fallback(title, year)
            state["requests_today"] += 1
            method = "fuzzy"

            if is_rate_limited(data):
                print("OMDb reports daily limit reached. Stopping cleanly.")
                state["last_index"] = i - 1
                save_progress(state)
                break

        if data is None or data.get("Response") != "True":
            unmatched_writer.writerow({"name": title, "year": year, "reason": "no match found"})
        else:
            enriched_writer.writerow({
                "name": title,
                "year": year,
                "match_method": method,
                "omdb_title": data.get("Title", ""),
                "omdb_year": data.get("Year", ""),
                "Plot": data.get("Plot", ""),
                "Genre_full": data.get("Genre", ""),
                "Actors_full": data.get("Actors", ""),
                "Writer_full": data.get("Writer", ""),
                "Director_full": data.get("Director", ""),
                "imdbRating_omdb": data.get("imdbRating", ""),
            })

        state["last_index"] = i
        processed_this_run += 1
        if processed_this_run % 50 == 0:
            save_progress(state)
            enriched_f.flush()
            unmatched_f.flush()
            print(f"...processed {processed_this_run} rows this run "
                  f"(at row {i}/{len(rows)})")

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    save_progress(state)
    enriched_f.close()
    unmatched_f.close()

    print(f"\nDone for now. Processed {processed_this_run} rows this run.")
    print(f"Overall progress: {state['last_index'] + 1}/{len(rows)} rows.")
    if state["last_index"] + 1 < len(rows):
        print("Run the script again (same command) to continue.")
    else:
        print("All rows processed! Check omdb_unmatched.csv for anything that never matched.")


if __name__ == "__main__":
    main()


Resuming from row 3778/7668. 0/950 requests used today.
...processed 50 rows this run (at row 3827/7668)
...processed 100 rows this run (at row 3877/7668)
...processed 150 rows this run (at row 3927/7668)
...processed 200 rows this run (at row 3977/7668)
...processed 250 rows this run (at row 4027/7668)
...processed 300 rows this run (at row 4077/7668)
...processed 350 rows this run (at row 4127/7668)
...processed 400 rows this run (at row 4177/7668)
...processed 450 rows this run (at row 4227/7668)
...processed 500 rows this run (at row 4277/7668)
...processed 550 rows this run (at row 4327/7668)
...processed 600 rows this run (at row 4377/7668)
...processed 650 rows this run (at row 4427/7668)
...processed 700 rows this run (at row 4477/7668)
...processed 750 rows this run (at row 4527/7668)
...processed 800 rows this run (at row 4577/7668)
...processed 850 rows this run (at row 4627/7668)
...processed 900 rows this run (at row 4677/7668)
Hit today's request budget (950). Stopping cl

In [2]:
import os
os.getcwd()

'C:\\Users\\Asus\\Downloads\\Jupiter Py Pro\\IMDB Project\\ML- Layer'

In [3]:
import pandas as pd
enriched = pd.read_csv(r"C:\Users\Asus\Downloads\Jupiter Py Pro\IMDB Project\ML- Layer\imdb_enriched.csv")
unmatched = pd.read_csv(r"C:\Users\Asus\Downloads\Jupiter Py Pro\IMDB Project\ML- Layer\omdb_unmatched.csv")

print(f"Matched: {len(enriched)}, Unmatched: {len(unmatched)}")
print(f"Match rate: {len(enriched) / (len(enriched) + len(unmatched)) * 100:.1f}%")
print()
print(enriched['Plot'].str.split().str.len().describe())  # word count stats — should be noticeably higher than the ~15-25 word short plots from before

Matched: 4689, Unmatched: 26
Match rate: 99.4%

count    4665.000000
mean       97.139979
std        64.343611
min         8.000000
25%        56.000000
50%        81.000000
75%       117.000000
max       559.000000
Name: Plot, dtype: float64
